# Editing analysis

Compilation of editing results and generation of MLE for each sensor.

In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats
import os
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

In [9]:
sample_key = pd.read_excel('../../screening_data/sample_key.xlsx')

#exclude SY-5609 since this is same as CBE subpool 1
sample_key = sample_key[sample_key['Screen_data_location']!='SY_5609'].reset_index(drop=True)

In [13]:
#define the editing files that match up with the samples (2 for each)
file1 = []
file2 = []

fp = '../../screening_data'

for i, val in sample_key.iterrows():
    fn = val['File_Name']
    loc = val['Screen_data_location']

    fp2 = f'{fp}/{loc}_screen_data/editing'

    files = os.listdir(fp2)
    files_relevant = sorted([i for i in files if fn in i])

    file1.append(f'{fp2}/{files_relevant[0]}')
    file2.append(f'{fp2}/{files_relevant[1]}')

sample_key['file1'] = file1
sample_key['file2'] = file2

sample_key

,File_Name,Sample,Editor,Screen_data_location,Subpool,file1,file2
0,D24-257001,Plasmid,PLASMID,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
1,D24-257002,T0_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
2,D24-257003,T0_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
3,D24-257004,DMSO_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
4,D24-257005,DMSO_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...
...,...,...,...,...,...,...,...
99,D25-188027,ABE_CDK12-IN-2_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...
100,D25-188028,ABE_HQ461_REP1,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...
101,D25-188029,ABE_HQ461_REP2,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...
102,D25-188030,ABE_HQ461_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...


In [22]:
#and create groups for MLE calculation
g = []
for i, val in sample_key.iterrows():
    name = val['Sample']
    if name not in ['Plasmid', 'PLASMID_LIBRARY']:
        g.append(name[:-5])
    elif name in ['Plasmid', 'PLASMID_LIBRARY']:
        g.append('Plasmid')

sample_key['group'] = g
sample_key

,File_Name,Sample,Editor,Screen_data_location,Subpool,file1,file2,group
0,D24-257001,Plasmid,PLASMID,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,Plasmid
1,D24-257002,T0_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,T0
2,D24-257003,T0_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,T0
3,D24-257004,DMSO_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
4,D24-257005,DMSO_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
...,...,...,...,...,...,...,...,...
99,D25-188027,ABE_CDK12-IN-2_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_CDK12-IN-2
100,D25-188028,ABE_HQ461_REP1,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_HQ461
101,D25-188029,ABE_HQ461_REP2,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_HQ461
102,D25-188030,ABE_HQ461_REP3,ABE,CDK12_13,2,../../screening_data/CDK12_13_screen_data/edit...,../../screening_data/CDK12_13_screen_data/edit...,ABE_HQ461


In [54]:
def HGVSp_combiner(subset):

    file_name_list = []
    for i, val in subset.iterrows():
        f1 = val['file1']
        f2 = val['file2']
        file_name_list.append(f1)
        file_name_list.append(f2)

    df_holder = []
    for k in file_name_list:
        d = pd.read_csv(k)
        df_holder.append(d)

    #and the combining
    new_out = pd.concat(df_holder).groupby(['Edit', 'HGVSp', 'Num_edits', 'DNA Change', 'Canonical_edit', 'Canonical_window', 'gRNA_id']).sum().reset_index().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    hgvsp_only = new_out[['gRNA_id', 'HGVSp', '#Reads']].groupby(['gRNA_id', 'HGVSp'], as_index=False).sum().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    #and calculate percentages efficiently
    new_out['%Reads'] = (new_out['#Reads'] / new_out.groupby('gRNA_id')['#Reads'].transform('sum')) * 100

    hgvsp_only['%Reads'] = (hgvsp_only['#Reads'] / hgvsp_only.groupby('gRNA_id')['#Reads'].transform('sum')) * 100


    return new_out, hgvsp_only

In [52]:
a2 = df[df['gRNA_id']==np.unique(df['gRNA_id'])[100]]
print(sum(a2['#Reads']))
a2

10266


,gRNA_id,HGVSp,#Reads,%Reads
13292,gRNA_CDK19_targ_6373,A395T,5964,58.094681
13293,gRNA_CDK19_targ_6373,WT,2025,19.725307
13294,gRNA_CDK19_targ_6373,A391S_P393S_Q394T_A395D,547,5.328268
13295,gRNA_CDK19_targ_6373,A391T_P392D_P393T_Q394R_A395G_P396T,472,4.597701
13296,gRNA_CDK19_targ_6373,P392A_Q394P_A395P,379,3.691798
...,...,...,...,...
13374,gRNA_CDK19_targ_6373,P397Q,1,0.009741
13375,gRNA_CDK19_targ_6373,P397T,1,0.009741
13376,gRNA_CDK19_targ_6373,Q394*_A395T,1,0.009741
13377,gRNA_CDK19_targ_6373,Q394P,1,0.009741


In [53]:
5964/10266

0.5809468147282291

In [ ]:
#need to group together subpool because of redo of T0_REP3 for CBE in Subpool1 (sequenced with ABE)
s = [['ABE_subpool_1', 'CBE_subpool_1'], ['CDK12_13'], ['KB_compound_mut']]

for i in s:
    subset = sample_key[sample_key['Screen_data_location'].isin(i)]

    CBE_portion = subset[subset['Editor']=='CBE']
    ABE_portion = subset[subset['Editor']=='ABE']
    Plasmid = subset[subset['Editor']=='PLASMID']

    CBE_groups = np.unique(CBE_portion['group'])
    ABE_groups = np.unique(ABE_portion['group'])
    Plasmid_groups = np.unique(Plasmid['group'])

    for k in CBE_groups:
        subset = CBE_portion[CBE_portion['group']==k]['Sample']

    for j in ABE_groups:
        subset = ABE_portion[ABE_portion['group']==j]['Sample']

    for b in Plasmid_groups:

        subset = Plasmid[Plasmid['group']==b]['Sample']

CBE
CBE
CBE
CBE


In [29]:
CBE_portion

,File_Name,Sample,Editor,Screen_data_location,Subpool,file1,file2,group
1,D24-257002,T0_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,T0
2,D24-257003,T0_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,T0
3,D24-257004,DMSO_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
4,D24-257005,DMSO_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
5,D24-257006,DMSO_REP3,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,DMSO
6,D24-257007,KI-CDK9d-32_100nM_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,KI-CDK9d-32_100nM
7,D24-257008,KI-CDK9d-32_100nM_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,KI-CDK9d-32_100nM
8,D24-257009,KI-CDK9d-32_100nM_REP3,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,KI-CDK9d-32_100nM
9,D24-257010,KI-CDK9d-32_1000nM_REP1,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,KI-CDK9d-32_1000nM
10,D24-257011,KI-CDK9d-32_1000nM_REP2,CBE,CBE_subpool_1,1,../../screening_data/CBE_subpool_1_screen_data...,../../screening_data/CBE_subpool_1_screen_data...,KI-CDK9d-32_1000nM


In [ ]:
'D24-295001-1-7018H' '_guide_split_BARCODES_HGVSp_sensor_quant_v2'

In [ ]:
def HGVSp_combiner(file_name_list):

    fp = 'CBE_editing/HGVSp_quant_full_v2'

    df_holder = []
    for k in file_name_list:
        d = pd.read_csv(f'{fp}/{k}')
        df_holder.append(d)

    #and the combining
    new_out = pd.concat(df_holder).groupby(['Edit', 'HGVSp', 'Num_edits', 'DNA Change', 'Canonical_edit', 'Canonical_window', 'gRNA_id']).sum().reset_index().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    hgvsp_only = new_out[['gRNA_id', 'HGVSp', '#Reads']].groupby(['gRNA_id', 'HGVSp'], as_index=False).sum().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    return new_out, hgvsp_only

In [ ]:
output = sorted(os.listdir('CBE_editing/HGVSp_quant_full_v2'))

name_dict = {'D24-257001' : 'Plasmid',
'D24-257002' : 'T0_REP1',
'D24-257003' : 'T0_REP2',
'D24-257004' : 'DMSO_REP1', 
'D24-257005' : 'DMSO_REP2', 
'D24-257006' : 'DMSO_REP3', 
'D24-257007' : 'KI-CDK9d-32_100nM_REP1', 
'D24-257008' : 'KI-CDK9d-32_100nM_REP2',
'D24-257009' : 'KI-CDK9d-32_100nM_REP3',
'D24-257010' : 'KI-CDK9d-32_1000nM_REP1', 
'D24-257011' : 'KI-CDK9d-32_1000nM_REP2', 
'D24-257012' : 'KI-CDK9d-32_1000nM_REP3', 
'D24-257013' : 'KI-CDK9d-32N_1250nM_REP1', 
'D24-257014' : 'KI-CDK9d-32N_1250nM_REP2', 
'D24-257015' : 'KI-CDK9d-32N_1250nM_REP3', 
'D24-257016' : 'KI-CDK9d-32N_5000nM_REP1', 
'D24-257017' : 'KI-CDK9d-32N_5000nM_REP2', 
'D24-257018' : 'KI-CDK9d-32N_5000nM_REP3', 
'D24-257019' : 'KB-0742_1500nM_REP1', 
'D24-257020' : 'KB-0742_1500nM_REP2', 
'D24-257021' : 'KB-0742_1500nM_REP3', 
'D24-257022' : 'Senexin_B_2000nM_REP1', 
'D24-257023' : 'Senexin_B_2000nM_REP2',
'D24-257024' : 'Senexin_B_2000nM_REP3',
'D24-257025' : 'Senexin_B_15000nM_REP1',
'D24-257026' : 'Senexin_B_15000nM_REP2',
'D24-257027' : 'Senexin_B_15000nM_REP3', 
'D24-257028' : 'SEL120_4000nM_REP1',
'D24-257029' : 'SEL120_4000nM_REP2', 
'D24-257030' : 'SEL120_4000nM_REP3',}

plasmid = output[:2]

all_samples = output[2:]


def HGVSp_combiner(file_name_list):

    fp = 'CBE_editing/HGVSp_quant_full_v2'

    df_holder = []
    for k in file_name_list:
        d = pd.read_csv(f'{fp}/{k}')
        df_holder.append(d)

    #and the combining
    new_out = pd.concat(df_holder).groupby(['Edit', 'HGVSp', 'Num_edits', 'DNA Change', 'Canonical_edit', 'Canonical_window', 'gRNA_id']).sum().reset_index().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    hgvsp_only = new_out[['gRNA_id', 'HGVSp', '#Reads']].groupby(['gRNA_id', 'HGVSp'], as_index=False).sum().sort_values(['gRNA_id', '#Reads'], ascending=[True, False]).reset_index(drop=True)

    return new_out, hgvsp_only


plasmid_combined, plasmid_hgvsp_only = HGVSp_combiner(plasmid)
all_combined, all_hgvsp_only = HGVSp_combiner(all_samples)

In [ ]:
#and adding percentage information
#then generate the percentages for each gRNA
def perc_adder(list_of_gRNAs, df):

    holder = []
    for i in list_of_gRNAs:

        sub = df[df['gRNA_id']==i]
        if len(sub)>0:
            sub['%Reads'] = 100*sub['#Reads']/sum(sub['#Reads'])
            holder.append(sub)

    df_new = pd.concat(holder).reset_index(drop=True)

    return df_new

lib_cluster = pd.read_csv('cluster_scripts/CDK_library_CLUSTER.csv')
targ_pool1_cluster = lib_cluster[(lib_cluster['Pool']=='F1-R1') & (lib_cluster['classification']=='targeting')].reset_index(drop=True)
list_of_gRNAs = list(targ_pool1_cluster['gRNA_id'])



all_combined2 = perc_adder(list_of_gRNAs, all_combined)
all_hgvsp_only2 = perc_adder(list_of_gRNAs, all_hgvsp_only)
plasmid_combined2 = perc_adder(list_of_gRNAs, plasmid_combined)
plasmid_hgvsp_only2 = perc_adder(list_of_gRNAs, plasmid_hgvsp_only)

In [ ]:
#saving tables as zip files (for storage considerations)
filename = 'CBE_editing/all_samples_combined_edit'
compression_options = dict(method='zip', archive_name=f'{filename}.csv')
#all_combined2.to_csv(f'{filename}.zip', compression=compression_options, index=False)

filename = 'CBE_editing/all_samples_combined_HGVSp_only'
compression_options = dict(method='zip', archive_name=f'{filename}.csv')
#all_hgvsp_only2.to_csv(f'{filename}.zip', compression=compression_options, index=False)

filename = 'CBE_editing/plasmid_combined_edit'
compression_options = dict(method='zip', archive_name=f'{filename}.csv')
#plasmid_combined2.to_csv(f'{filename}.zip', compression=compression_options, index=False)

filename = 'CBE_editing/plasmid_combined_HGVSp_only'
compression_options = dict(method='zip', archive_name=f'{filename}.csv')
#plasmid_hgvsp_only2.to_csv(f'{filename}.zip', compression=compression_options, index=False)